# LangChain + Groq Tutorial (Step by Step)

## Requirements
For this notebook, you need:
1. `langchain`, `langchain-groq`, and `python-dotenv`.
2. A free Groq account and an API key.
3. `GROQ_API_KEY` available from `.env` or from a secure prompt.

Optional in PowerShell:
`$env:GROQ_API_KEY="your_api_key_here"`

In [1]:
%pip install -U langchain langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import getpass
import importlib

# Load .env if python-dotenv is available
dotenv_spec = importlib.util.find_spec("dotenv")
if dotenv_spec is not None:
    dotenv_module = importlib.import_module("dotenv")
    dotenv_module.load_dotenv()

def _clean_key(value: str | None) -> str:
    if not value:
        return ""
    return value.strip().strip("\"").strip("'")

# Ask for the key only if it is still missing
if not os.getenv("GROQ_API_KEY"):
    try:
        entered_key = getpass.getpass("Enter GROQ_API_KEY: ")
    except Exception:
        entered_key = input("Enter GROQ_API_KEY (visible): ")
    os.environ["GROQ_API_KEY"] = _clean_key(entered_key)

if not _clean_key(os.getenv("GROQ_API_KEY")):
    raise EnvironmentError("GROQ_API_KEY not found. Add it to .env or enter it when prompted.")

print("[✓] GROQ_API_KEY detected")

OSError: GROQ_API_KEY not found. Add it to .env or enter it when prompted.

## 1) Initialize the model
Create a chat model instance that will be reused across the tutorial.

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

print("[✓] Model initialized: llama-3.1-8b-instant")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 29.201196857s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

## 2) Direct invocation with messages
Send a system instruction and user message directly to the model.

This step shows the most basic way to call a chat model in LangChain.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English into Spanish."),
    HumanMessage(content="Hello! How are you doing today?"),
]

response = model.invoke(messages)
print(response.content)

"You are an expert weather forecaster, who speaks in puns.\n\nYou have access to two tools:\n\n- get_weather_for_location: use this to get the weather for a specific location\n- get_user_location: use this to get the user's location\n\nIf a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."

## 3) Add an output parser
A parser normalizes model output into a plain string for easier downstream use.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
chain_with_parser = model | parser

parsed_text = chain_with_parser.invoke(messages)
print(parsed_text)

## 4) Build prompt templates
Prompt templates let you parameterize instructions (for language, input text, style, etc.).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following into {language}."),
    ("user", "{text}"),
])

prompt_value = template.invoke({"language": "French", "text": "Good morning!"})
for message in prompt_value.messages:
    print(f"[{message.__class__.__name__}] {message.content}")

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.0-flash', temperature=0.5, max_output_tokens=1000, timeout=10.0, client=<google.genai.client.Client object at 0x00000165B50810F0>, default_metadata=(), model_kwargs={})

## 5) Compose the full chain
Combine `template | model | parser` for a reusable translation pipeline.

In [ ]:
full_chain = template | model | parser

examples = [
    {"language": "German", "text": "I love programming!"},
    {"language": "Italian", "text": "The weather is beautiful today."},
]

for item in examples:
    out = full_chain.invoke(item)
    print(f"{item['language']}: {out}")

## 6) Stream model output
Streaming is useful for chat-like UX because tokens appear as they are generated.

In [ ]:
print("Output: ", end="")
for chunk in full_chain.stream({"language": "Portuguese", "text": "Artificial Intelligence is changing the world."}):
    print(chunk, end="", flush=True)
print()

## Next step
From here you can adapt the same pattern to a RAG pipeline by replacing the user text with retrieved context + question.

In [ ]:
print("Tutorial completed ")

Gemini request failed (usually quota/billing).
Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 57.310205586s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini

TypeError: 'StructuredTool' object is not callable

In [ ]:
# Optional quick check
full_chain.invoke({"language": "Spanish", "text": "See you tomorrow!"})